[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C64_ML_Knowledge_QA_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境（三段式答法 / 60秒-3分钟估算器 / "我不知道"检查器 / 知识地图与间隔重复）

目标：把"怎么讲清楚一个知识点"从一堆经验之谈，变成**几个可以运行、可以断言的小工具**。

本 notebook 你会亲手实现：
1. **环境自检** —— 确认 Python / numpy 可用（本课全程不需要 GPU、不需要联网）
2. **三段式数据结构与格式化器** —— 一句话定义 → 为什么需要 → 什么时候失效，拼成 60 秒/3 分钟两档
3. **60秒/3分钟时长估算器** —— 用字符数估算讲话时长，别在面试里超时
4. **"我不知道"合规检查器** —— 承认 + 会怎么查/推 + 给相邻已知，三段缺一不可
5. **知识地图与掌握度追踪** —— 把 C64 全课程结构变成你自己的复习进度表
6. **间隔重复调度器（SM-2 简化版）** —— 答得越轻松间隔越长，答错了清零重来

> 心智模型：**这一环节考的不是"你知道多少"，是"你能不能把已经知道的东西讲清楚，并且诚实地标出你不知道的地方"。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'argsort')

print('\n✅ 环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · 三段式数据结构与格式化器

把"一句话定义 / 为什么需要 / 什么时候失效"变成一个字典 + 两个格式化函数。

In [ ]:
def make_answer(term, one_liner, why, failure):
    """三段式知识点：一句话定义 / 为什么需要 / 什么时候失效。"""
    return {'term': term, 'one_liner': one_liner, 'why': why, 'failure': failure}

def format_60s(answer):
    """60 秒版：定义 + 为什么 + 一条失效边界，直接拼接。"""
    a = answer
    return f"{a['one_liner']} {a['why']} {a['failure']}"

def format_3min(answer, extra):
    """3 分钟版：60 秒版 + 追加的机制细节/反例（extra 是一句话）。"""
    return format_60s(answer) + ' ' + extra

overfitting = make_answer(
    term='过拟合 overfitting',
    one_liner='过拟合是模型在训练数据上表现好，但在没见过的数据上明显更差的现象。',
    why='几乎所有模型选择、正则化、早停的决策，本质上都是在用训练表现代理真实泛化能力，不理解它就没法解释为什么要留验证集。',
    failure='数据量足够大或模型有强隐式正则时，训练/验证差距可能远小于直觉，甚至出现"更大的模型反而更不容易过拟合"的双下降现象（见模块01）。',
)

s60 = format_60s(overfitting)
s3m = format_3min(overfitting, '如果要展开，我会补充偏差-方差分解的正式框架，以及 L2/dropout/early stopping 这三种机制虽然效果相近但原理完全不同。')

print('【60秒版】', s60)
print()
print('【3分钟版】', s3m)

assert len(s60) < len(s3m)
assert overfitting['term'] == '过拟合 overfitting'
assert all(k in overfitting for k in ['one_liner', 'why', 'failure'])
print('\n✅ 三段式结构就位。')

## 2 · 60秒/3分钟时长估算器

依据：中文技术讲解的经验语速约 **250 字/分钟**（比日常聊天略慢，因为夹杂术语与停顿）。
用确定长度的合成字符串验证阈值，避免手数中文字符引入错误。

In [ ]:
def speaking_seconds(text, cpm=250):
    """按中文口语语速估算讲完这段文字需要多少秒（cpm=每分钟字数）。"""
    return len(text) / cpm * 60

def classify_length(text, cpm=250):
    s = speaking_seconds(text, cpm)
    if s <= 75:        # 60秒 + 25% 缓冲
        return '60s'
    if s <= 200:       # 3分钟 + 约10秒缓冲
        return '3min'
    return 'too_long'

assert classify_length('x' * 250) == '60s'       # 60.0s
assert classify_length('x' * 310) == '60s'       # 74.4s，仍在缓冲区内
assert classify_length('x' * 320) == '3min'      # 76.8s
assert classify_length('x' * 800) == '3min'      # 192.0s
assert classify_length('x' * 850) == 'too_long'  # 204.0s

print('60秒版实际用时 :', round(speaking_seconds(s60), 1), 's ->', classify_length(s60))
print('3分钟版实际用时:', round(speaking_seconds(s3m), 1), 's ->', classify_length(s3m))
print('\n✅ 阈值就位：250字/分钟是经验语速，超过约320字就已经超出60秒的安全区。')

## 3 · "我不知道"合规检查器

三段缺一不可：承认边界 / 会怎么查或推 / 给一个相邻已知点。

In [ ]:
def check_honest_unknown(answer):
    """检查『我不知道』的三段式是否完整。返回缺失字段列表（空列表表示合规）。"""
    required = ['acknowledge', 'method', 'adjacent']
    return [k for k in required if not str(answer.get(k, '')).strip()]

good = {
    'acknowledge': '没有直接用过 Conformal Prediction',
    'method': '从名字判断它跟"给预测配一个有统计保证的置信区间"有关，真要用我会先查它对样本可交换性的假设',
    'adjacent': '我们现有的温度缩放校准流程（见模块04）应该可以作为对照起点',
}
bad = {'acknowledge': '不太清楚'}   # 只有承认，没有方法也没有锚点

assert check_honest_unknown(good) == []
assert check_honest_unknown(bad) == ['method', 'adjacent']
assert check_honest_unknown({}) == ['acknowledge', 'method', 'adjacent']

print('合规回答缺失项       :', check_honest_unknown(good))
print('硬编/沉默型回答缺失项:', check_honest_unknown(bad))
print('\n✅ "我不知道"检查器就位：诚实度不是靠说"不知道"三个字拿到的，是靠后面两段拿到的。')

## 4 · 知识地图与掌握度追踪

把 C64 全课程结构（模块 01-05）本身做成知识地图，每个子点带一个 0-5 的掌握度。

In [ ]:
KNOWLEDGE_MAP = {
    'm01_ml_basics':     ['偏差方差', '正则化', '交叉验证', '类别不平衡', '集成方法', '生成判别', '维度灾难'],
    'm02_optimization':  ['优化器', 'weight_decay', '学习率调度', '初始化', '混合精度'],
    'm03_architectures': ['卷积', '归一化', '残差连接', 'attention', 'CNN_vs_Transformer'],
    'm04_eval_stats':    ['指标选择', '校准', '置信区间', 'AB测试', '统计陷阱'],
}

# 掌握度 0-5，0 表示还没学过
MASTERY = {
    '偏差方差': 4, '正则化': 3, '交叉验证': 3, '类别不平衡': 2, '集成方法': 3,
    '生成判别': 1, '维度灾难': 2, '优化器': 3, 'weight_decay': 2,
    '学习率调度': 2, '初始化': 1, '混合精度': 0, '卷积': 3, '归一化': 2,
    '残差连接': 3, 'attention': 2, 'CNN_vs_Transformer': 2,
    '指标选择': 3, '校准': 1, '置信区间': 2, 'AB测试': 2, '统计陷阱': 2,
}

def weak_topics(mastery, threshold=3):
    """掌握度 < threshold 的子点，按掌握度升序、同分按名字排序。"""
    weak = [(k, v) for k, v in mastery.items() if v < threshold]
    return sorted(weak, key=lambda kv: (kv[1], kv[0]))

all_topics = sum(KNOWLEDGE_MAP.values(), [])
assert set(all_topics) == set(MASTERY.keys())
w = weak_topics(MASTERY)
assert w[0] == ('混合精度', 0)
assert all(v < 3 for _, v in w)

print(f'知识地图覆盖 {len(all_topics)} 个子点，来自 {len(KNOWLEDGE_MAP)} 个后续模块。')
print('薄弱环节（掌握度<3，优先复习）：')
for k, v in w:
    print(f'  {k:<20} 掌握度={v}')
print('\n✅ 知识地图就位：它同时是本课程 m01-m05 的内容地图，也是你自测进度的追踪器。')

## 5 · 间隔重复调度器（SM-2 简化版）

`quality` 是你对刚才回忆这条知识点的自评（0-5，<3 视为没想起来/答错）。

In [ ]:
def sm2_update(quality, reps, ease, interval):
    """SM-2 简化版。返回 (new_interval, new_reps, new_ease)。"""
    if quality < 3:
        new_reps, new_interval = 0, 1
    else:
        if reps == 0:
            new_interval = 1
        elif reps == 1:
            new_interval = 6
        else:
            new_interval = round(interval * ease)
        new_reps = reps + 1
    new_ease = ease + (0.1 - (5 - quality) * (0.08 + (5 - quality) * 0.02))
    new_ease = max(new_ease, 1.3)
    return new_interval, new_reps, new_ease

# 连续 4 次都答得很轻松（quality=5）
interval, reps, ease = 0, 0, 2.5
trace = []
for q in [5, 5, 5, 5]:
    interval, reps, ease = sm2_update(q, reps, ease, interval)
    trace.append((interval, reps, round(ease, 4)))
assert trace == [(1, 1, 2.6), (6, 2, 2.7), (16, 3, 2.8), (45, 4, 2.9)]

# 前两次轻松，第三次没想起来（quality=2）—— 间隔清零重来
interval, reps, ease = 0, 0, 2.5
trace2 = []
for q in [5, 5, 2]:
    interval, reps, ease = sm2_update(q, reps, ease, interval)
    trace2.append((interval, reps, round(ease, 4)))
assert trace2 == [(1, 1, 2.6), (6, 2, 2.7), (1, 0, 2.38)]

print('连续答对 4 次的复习间隔演化(天):', [t[0] for t in trace])
print('第 3 次答错后的间隔演化(天)     :', [t[0] for t in trace2])
print('\n✅ 间隔重复调度器就位：答得越轻松间隔拉得越长；一旦答错，间隔清零重来。')

## ✏️ 练习 1：三段式完整性检查器

实现 `check_three_part(answer)`，检查一个 `make_answer()` 风格的字典是否给全了三段式的三个字段
`['one_liner', 'why', 'failure']`（顺序固定），空字符串/纯空白/缺失都算不合格。
返回**缺失字段名的列表**，按上面给的顺序。

In [ ]:
def check_three_part(answer):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
complete = {'term': 'x', 'one_liner': '定义', 'why': '动机', 'failure': '边界'}
assert check_three_part(complete) == []

missing_why = {'term': 'x', 'one_liner': '定义', 'failure': '边界'}
assert check_three_part(missing_why) == ['why']

empty_failure = {'one_liner': '定义', 'why': '动机', 'failure': '   '}
assert check_three_part(empty_failure) == ['failure']

assert check_three_part({}) == ['one_liner', 'why', 'failure']

for name, ans in [('complete', complete), ('missing_why', missing_why), ('empty_failure', empty_failure)]:
    print(f'{name:<15} 缺失 -> {check_three_part(ans)}')
print('\n✅ 练习 1 通过：面试时自己在脑子里过一遍这三个字段，就是最快的自查表。')

## ✏️ 练习 2：批量调度器

实现 `schedule_reviews(qualities, ease=2.5, interval=0, reps=0)`：
依次对 `qualities` 里每个自评质量调用 `sm2_update`，返回**每次调用后的 interval 组成的列表**。

In [ ]:
def schedule_reviews(qualities, ease=2.5, interval=0, reps=0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert schedule_reviews([5, 5, 5, 5]) == [1, 6, 16, 45]
assert schedule_reviews([5, 5, 2]) == [1, 6, 1]
assert schedule_reviews([]) == []
assert schedule_reviews([1]) == [1]          # 第一次就没想起来：interval 清零为 1

print('连续 4 次轻松    ->', schedule_reviews([5, 5, 5, 5]))
print('第 3 次没想起来  ->', schedule_reviews([5, 5, 2]))
print('\n✅ 练习 2 通过：这就是知识地图节点背后真正在跑的调度逻辑。')

## ✏️ 练习 3：该复习哪个薄弱点

实现 `next_review_topic(mastery)`：返回掌握度**最低**的主题名；如果并列最低，
返回 **Unicode 码点序最小**的那个。

In [ ]:
def next_review_topic(mastery):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
m1 = {'偏差方差': 4, '正则化': 2, '交叉验证': 2, '集成方法': 5}
assert next_review_topic(m1) == '交叉验证'          # 2分并列，'交'<'正'（Unicode码点）

m2 = {'A': 3, 'B': 1, 'C': 5}
assert next_review_topic(m2) == 'B'

m3 = {'只有一个': 0}
assert next_review_topic(m3) == '只有一个'

print(next_review_topic(m1), next_review_topic(m2), next_review_topic(m3))
print('\n✅ 练习 3 通过。')

## ✏️ 练习 4：三段式答案的时长自检

实现 `answer_time_budget(one_liner, why, failure, cpm=250)`：把三段用**单个空格**拼接
（`f'{one_liner} {why} {failure}'`），套用 `classify_length` 返回档位。

In [ ]:
def answer_time_budget(one_liner, why, failure, cpm=250):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
short = answer_time_budget('x'*80, 'x'*80, 'x'*80, cpm=250)      # 拼接后242字符 -> 58.1s
assert short == '60s'

long3 = answer_time_budget('x'*250, 'x'*250, 'x'*250, cpm=250)   # 752字符 -> 180.5s
assert long3 == '3min'

toolong = answer_time_budget('x'*300, 'x'*300, 'x'*300, cpm=250) # 902字符 -> 216.5s
assert toolong == 'too_long'

print(short, long3, toolong)
print('\n✅ 练习 4 通过：真答题时把三段套进去，就知道自己是不是又讲超时了。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def check_three_part(answer):
    order = ['one_liner', 'why', 'failure']
    return [k for k in order if not str(answer.get(k, '')).strip()]

In [ ]:
# 练习 2 参考答案
def schedule_reviews(qualities, ease=2.5, interval=0, reps=0):
    out = []
    for q in qualities:
        interval, reps, ease = sm2_update(q, reps, ease, interval)
        out.append(interval)
    return out

In [ ]:
# 练习 3 参考答案
def next_review_topic(mastery):
    return sorted(mastery.items(), key=lambda kv: (kv[1], kv[0]))[0][0]

In [ ]:
# 练习 4 参考答案
def answer_time_budget(one_liner, why, failure, cpm=250):
    text = f'{one_liner} {why} {failure}'
    return classify_length(text, cpm=cpm)

---
## 🧪 真实工程胶囊：三段式速记卡 + 面试前自测流程

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 三段式速记卡（面试当天可以直接照这个顺序说）
# ══════════════════════════════════════════════════════════════════════
# 1) 一句话定义   —— 3秒内让对方确认"你说的是我问的那个东西"
# 2) 为什么需要   —— 证明你懂它解决什么问题，而不是死记硬背
# 3) 什么时候失效 —— 区分度最高的一步，主动说，不要等被问
#    面试官继续追问才展开到 3 分钟版：机制细节 + 一个具体反例/权衡
#
# ══════════════════════════════════════════════════════════════════════
# B. "我不知道"三连句（照抄即可）
# ══════════════════════════════════════════════════════════════════════
# ① 承认: "这个我没有直接实践过 / 不熟"
# ② 方法: "从名字/相关概念判断它应该是……；真要用我会先查……/ 用……去类比推一下"
# ③ 锚点: "但它跟我熟悉的……（举一个真正懂的相邻知识点）应该是同一类问题"
#
# ══════════════════════════════════════════════════════════════════════
# C. 面试前的自测流程（配合本课知识地图使用）
# ══════════════════════════════════════════════════════════════════════
# □ 把 m01-m05 的每个子点过一遍 weak_topics()，掌握度<3 的排进复习队列
# □ 对每个子点自问三句："我什么时候会用它" / "我什么时候不会用它" / "给我一个它失败的例子"
# □ 用 schedule_reviews() 记录每次自测的 quality，让间隔重复告诉你哪天该回来
# □ 面试前一晚：只看薄弱清单，不要临时学新概念
#
# ══════════════════════════════════════════════════════════════════════
# D. 与本课程其他部分的分工
# ══════════════════════════════════════════════════════════════════════
# · 数学推导（softmax梯度、贝叶斯、k-fold的完整推导）      -> C07（本课引用，不重复）
# · 本课 m01-m05：ML基础/优化训练/架构/评估统计/快问快答  -> C64（本课）
# · 长尾与类别不平衡的完整处理方案（本课m01只给问答骨架）  -> C58-01
# · 通用算法编码题 / ML系统设计 / 结构化沟通              -> C62 / C63 / C65
'''
print(RECIPE)
for token in ['一句话定义', '我不知道', 'weak_topics', 'C07', 'C58-01']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：三段式速记卡 / 诚实表达模板 / 自测流程 / 课程分工')

### 小结

- **这一环节考的不是「你知道多少」，是「你能不能把已经知道的东西讲清楚，并诚实标出你不知道的地方」。**
  四个真实评分维度是准确性/深度/边界意识/诚实度，其中**边界意识区分度最高**——
  能主动说出"什么时候它会失效"，几乎不可能靠背书获得。
- **三段式答法**：一句话定义 → 为什么需要 → 什么时候失效。60 秒版只给这三段；
  3 分钟版才展开机制细节和具体反例，且永远先给定义、不要先讲历史或打比方。
- **"我不知道"的正确说法是三段式，不是一个词**：①承认边界 ②你会怎么查/怎么推 ③给一个相邻已知点。
  硬编的风险是不对称的——编对了不加分，编错了会让面试官怀疑你之前所有的回答。
- **知识地图 + 间隔重复（SM-2）**：把 C64 的模块结构当成你自己的复习进度表，
  答得轻松就拉长间隔，答错就清零重来——诚实自评比"看起来复习得很勤"更重要。
- 本课与 **C07** 的边界：C07 推导，C64 讲清楚并接住追问；需要推导细节时统一写"见 C07-XX"。

下一站：**模块 01 · 机器学习基础问答** —— 把偏差-方差、正则化、交叉验证、集成方法这些
C07 已经推导过的内容，重新组织成三段式 + 追问的形式，并补上一个 C07 没讲的局限：深度学习的双下降。